### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [6]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

## Step 1: Load and split the dataset

In [4]:
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(raw_docs)

In [5]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

## Step 2: Vector Store

In [7]:
embedding_model = OllamaEmbeddings(model = "nomic-embed-text-v2-moe:latest")
vectorstore = FAISS.from_documents(chunks, embedding_model)


## Step 3: MMR Retriever

In [8]:
retriever = vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002D3F3BA5550>, search_type='mmr', search_kwargs={'k': 5})

## Step 4 : LLM and Prompt

In [9]:
llm = ChatOllama(model="gemma3:latest")
llm


ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, model='gemma3:latest')

## Query expansion

In [ ]:
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context. and I will again send this query to an LLM to retrieve relavant documents from a vector database. Dont't put unnecessary words in the expanded query.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context. and I will again send this query to an LLM to retrieve relavant documents from a vector database. Dont\'t put unnecessary words in the expanded query.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, model='gemma3:latest')
| StrOutputParser()

In [16]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'“Langchain memory modules,” “conversational memory,” “LLM memory management,” “memory chains,” “vector databases for Langchain,” “retrieval-augmented generation memory,” “knowledge graph memory Langchain,” “session memory Langchain,” “long-term memory Langchain,” “episodic memory Langchain,” “chunking memory Langchain”\n'

## RAG answering prompt

In [17]:
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm = llm, prompt = answer_prompt)

## Step 5: Full RAG pipeline with query expansion

In [18]:
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

## Step 6: Run query

In [19]:
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Here's an expanded query designed to improve document retrieval for the original question, incorporating synonyms, technical terms, and context:

**Expanded Query:** "{'input': 'What types of data stores and memory formats does LangChain integrate with, including vector databases, chromaDB, FAISS, and support for LlamaIndex memory modules?'}" 

**Reasoning for Changes:**

*   **"Data stores and memory formats"**:  More specific than just “memory,” capturing the broader scope of where LangChain manages information.
*   **"Integrate with"**:  Clarifies the intention – LangChain’s capabilities around data storage.
*   **"Vector databases, chromaDB, FAISS"**: Explicitly lists common vector database technologies LangChain utilizes, significantly increasing the chance of relevant documents appearing.
*   **"Support for LlamaIndex memory modules"**:  LangChain often works with LlamaIndex, and those modules use memory, so including this expands the search.

This expanded query provides the LLM

## Step 6: Run query

In [20]:
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

```json
{
  "input": "CrewAI agent capabilities, autonomous workforce solutions, virtual agent platforms, conversational AI, agent orchestration, task automation, digital workforce, human-AI collaboration, agent deployment, agent training, agent performance metrics, agent lifecycle management, robotic process automation (RPA) integration, natural language processing (NLP) agents"
}
```

✅ Answer:
 According to the context, CrewAI agents are defined with a purpose, a goal, and a set of tools they can use. They operate within a collaborative context, each with a defined role (such as researcher, planner, or executor) and contribute meaningfully to the overall crew objective.
